# **Tugas Besar Big Data**
## Implementasi Pipeline Big Data ETL dan ELT serta Dashboard Analitik pada dataset Medical Appointment No Shows
### Maleakhi Nymmo Augustus

## **Import Semua Packages/Library yang Digunakan**

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from google.colab import drive
import seaborn as sns
import gdown
import unicodedata
import time
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import os
import sqlite3

## **Konfigurasi Environment Mode**

In [ ]:
# KALO MAU DIKUMPUL UBAH KE FALSE YEAH BIAR GA DOUBLE
USE_GOOGLE_DRIVE = false

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Path ke folder project di GDrive
    PROJECT_ROOT = "/content/drive/MyDrive/Tugas_Besar_BigData_MaleakhiNymmo"
    print("Mode: Development (Tersimpan permanen di Google Drive)")
else:
    # Relative path untuk environment diluar
    PROJECT_ROOT = "."
    print("Mode: Production/Submission (Tersimpan sementara di lokal Colab)")

# Setup Folder Target
RAW_DIR = f"{PROJECT_ROOT}/raw"
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Folder raw siap di: {RAW_DIR}")

Mounted at /content/drive
Mode: Development (Tersimpan permanen di Google Drive)
Folder raw siap di: /content/drive/MyDrive/Tugas_Besar_BigData_MaleakhiNymmo/raw


## **Dataset**

Dataset utama yang digunakan dalam proyek ini adalah "Medical Appointment No Shows" yang diperoleh dari Kaggle

[KaggleV2-May-2016](https://https://drive.google.com/file/d/1c02QDzYeZ1tSMD6LDajXFNprMYWD-7bL/view?usp=drive_link)

Dataset ini berisi riwayat transaksi janji temu pasien di rumah sakit, mencakup informasi lokasi, kondisi kesehatan, dan status kehadiran (No-Show).

### `KaggleV2-May-2016.csv`



In [ ]:
print("--- MENYIAPKAN SOURCE 1 (APPOINTMENTS) ---")

nama_file_asli = 'KaggleV2-May-2016.csv'
file_id = '1c02QDzYeZ1tSMD6LDajXFNprMYWD-7bL'

# 1. Cek dan Unduh File Asli
if not os.path.exists(nama_file_asli):
    print("🔍 File lokal tidak ditemukan. Mencoba mengunduh dari Google Drive...")
    try:
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, nama_file_asli, quiet=False)
        print("✅ Berhasil mengunduh file dari Google Drive.")
    except Exception as e:
        print(f"Gagal mengunduh otomatis: {e}")
        print("Silakan upload file CSV manual ke panel Files di sebelah kiri.")
else:
    print("File asli ditemukan di penyimpanan lokal.")

# 2. Proses Pembuatan Data Simulasi Source 1
if os.path.exists(nama_file_asli):
    df_asli = pd.read_csv(nama_file_asli)

    print("\nINFORMASI DATA ASLI:")
    print(f"   - Total Baris: {df_asli.shape[0]}")
    print(f"   - Total Kolom: {df_asli.shape[1]}")
    print("   - Pratinjau Data:")
    display(df_asli.head(3))
    print("-" * 42 + "\n")

    # Copy data asli
    source1 = df_asli.copy()

    # Kotorin data sesuai skenario untuk tahap Transform nanti
    source1 = pd.concat([source1, source1.head(50)], ignore_index=True) # Tambah duplikat
    source1.loc[100:150, 'Age'] = np.nan
    source1.loc[0, 'Age'] = -5 # Buat outlier bawah
    source1.loc[1, 'Age'] = 250 # Buat outlier atas

    # Simpan sebagai CSV
    source1.to_csv('source1_appointments.csv', index=False)
    print("File 'source1_appointments.csv' BERHASIL DIBUAT (Dengan duplikat & anomali).")
else:
    print("Gagal membuat Source 1 karena file asli tidak ditemukan.")

--- MENYIAPKAN SOURCE 1 (APPOINTMENTS) ---
🔍 File lokal tidak ditemukan. Mencoba mengunduh dari Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1c02QDzYeZ1tSMD6LDajXFNprMYWD-7bL
To: /content/KaggleV2-May-2016.csv
100%|██████████| 10.7M/10.7M [00:00<00:00, 28.6MB/s]


✅ Berhasil mengunduh file dari Google Drive.

INFORMASI DATA ASLI:
   - Total Baris: 110527
   - Total Kolom: 14
   - Pratinjau Data:


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No


------------------------------------------

File 'source1_appointments.csv' BERHASIL DIBUAT (Dengan duplikat & anomali).


Penjelasan Dataset:

Dataset ini merupakan data utama yang berisi log janji temu pasien, yang memiliki Informasi sebagai berikut

    - Jumlah Baris Awal: 110.527 baris (Memenuhi syarat > 100.000 data).
    - Jumlah Kolom: 14 kolom (Memenuhi syarat minimal 12 kolom).

Rekayasa Kualitas data (Data Injection) dilakukan untuk mensimulasikan kondisi nyata dari data yang tidak bersih dan memenuhi syarat pengujian Data Cleaning pada Pipeline ETL. Data Injection meliputi:

    - Duplikasi: Menambahkan 50 baris duplikat untuk diuji pada tahap Deduplication.
    - Missing Values: Menghapus nilai pada kolom Age (Umur) sebanyak 50 data untuk diuji pada tahap Imputation.
    - Outliers: Memasukkan nilai yang ekstrim pada kolom Age (umur) untuk diuji pada tahap Outlier Detection.



### `source2_regions.json`

Dataset Sekunder yang digunakan dalam proyek ini adalah data resmi Region Vitória yang dibuat secara sintesis

Dataset ini berisi seluruh wilayah dari kota Vitória di Brazil.

Disimpan dalam format JSON untuk memenuhi syarat minimal dua sumber dan format yang berbeda

In [ ]:
print("--- MENYIAPKAN SOURCE 2 (WILAYAH) ---")

nama_file_asli = 'KaggleV2-May-2016.csv'

if os.path.exists(nama_file_asli):
    df_asli = pd.read_csv(nama_file_asli)

    # Mapping Resmi Region Vitória
    OFFICIAL_REGIONS_MAP = {
        "Região 1": ["CENTRO", "DO MOSCOSO", "FONTE GRANDE", "ILHA DO PRÍNCIPE", "PARQUE MOSCOSO", "PIEDADE", "SANTA CLARA", "VILA RUBIM"],
        "Região 2": ["ARIOVALDO FAVALESSA", "BELA VISTA", "CARATOÍRA", "DO CABRAL", "DO QUADRO", "ESTRELINHA", "GRANDE VITÓRIA", "INHANGUETÁ", "MÁRIO CYPRESTE", "SANTO ANTÔNIO", "SANTA TEREZA", "UNIVERSITÁRIO"],
        "Região 3": ["BENTO FERREIRA", "CONSOLAÇÃO", "CRUZAMENTO", "DE LOURDES", "FORTE SÃO JOÃO", "FRADINHOS", "GURIGICA", "HORTO", "ILHA DE SANTA MARIA", "JESUS DE NAZARETH", "JUCUTUQUARA", "MONTE BELO", "NAZARETH", "ROMÃO"],
        "Região 4": ["ANDORINHAS", "BONFIM", "DA PENHA", "ITARARÉ", "JOANA D'ARC", "MARUÍPE", "SANTA CECÍLIA", "SANTA MARTHA", "SANTOS DUMONT", "SÃO BENEDITO", "SÃO CRISTÓVÃO", "TABUAZEIRO"],
        "Região 5": ["BARRO VERMELHO", "ENSEADA DO SUÁ", "ILHA DO BOI", "ILHA DO FRADE", "PRAIA DO CANTO", "PRAIA DO SUÁ", "SANTA HELENA", "SANTA LÚCIA", "SANTA LUÍZA"],
        "Região 6": ["AEROPORTO", "ANTÔNIO HONÓRIO", "GOIABEIRAS", "JABOUR", "MARIA ORTIZ", "SEGURANÇA DO LAR", "SOLON BORGES"],
        "Região 7": ["COMDUSA", "CONQUISTA", "ILHA DAS CAIEIRAS", "NOVA PALESTINA", "REDENÇÃO", "RESISTÊNCIA", "SANTO ANDRÉ", "SANTOS REIS", "SÃO JOSÉ", "SÃO PEDRO"],
        "Região 8": ["JARDIM CAMBURI", "PARQUE INDUSTRIAL"],
        "Região 9": ["BOA VISTA", "JARDIM DA PENHA", "MATA DA PRAIA", "MORADA DE CAMBURI", "PONTAL DE CAMBURI", "REPÚBLICA"]
    }

    # Fungsi pembersih teks
    def normalize_text(text):
        if not text: return ""
        return ''.join(c for c in unicodedata.normalize('NFD', str(text))
                      if unicodedata.category(c) != 'Mn').upper().strip()

    # Fungsi pencocokan wilayah
    def get_official_region(location_name):
        norm_name = normalize_text(location_name)
        for region, neighborhoods in OFFICIAL_REGIONS_MAP.items():
            if any(norm_name == normalize_text(n) for n in neighborhoods):
                return region
        return "Outros/Luar Vitória"

    # Ambil kecamatan unik dan petakan
    kecamatan = df_asli['Neighbourhood'].unique()
    region_list = [get_official_region(loc) for loc in kecamatan]
    source2 = pd.DataFrame({'Lokasi': kecamatan, 'Region': region_list})

    # Simpan sebagai JSON
    source2.to_json('source2_regions.json', orient='records')
    print("File 'source2_regions.json' BERHASIL DIBUAT (Validasi Spasial Resmi).")

    # Menampilkan informasi tabel
    print("\nINFORMASI DATA SOURCE 2 (WILAYAH):")
    print(f"   - Total Lokasi Unik: {source2.shape[0]} Bairros")
    print(f"   - Distribusi Region:\n{source2['Region'].value_counts().to_string()}")
    print("\n   - Pratinjau Data Wilayah:")
    display(source2.head(5))
    print("-" * 42 + "\n")

else:
    print("Gagal membuat Source 2 karena file asli tidak ditemukan. Jalankan Cell 1A dulu.")

--- MENYIAPKAN SOURCE 2 (WILAYAH) ---
File 'source2_regions.json' BERHASIL DIBUAT (Validasi Spasial Resmi).

INFORMASI DATA SOURCE 2 (WILAYAH):
   - Total Lokasi Unik: 81 Bairros
   - Distribusi Region:
Region
Região 3               14
Região 2               12
Região 4               11
Região 7               10
Região 5                9
Região 1                8
Região 6                7
Região 9                6
Região 8                2
Outros/Luar Vitória     2

   - Pratinjau Data Wilayah:


,Lokasi,Region
0,JARDIM DA PENHA,Região 9
1,MATA DA PRAIA,Região 9
2,PONTAL DE CAMBURI,Região 9
3,REPÚBLICA,Região 9
4,GOIABEIRAS,Região 6


------------------------------------------



### Tabel Ketentuan Spesifikasi Dataset

| Ketentuan | Status | Keterangan |
| :--- | :--- | :--- |
| **Minimal 100.000 Baris** | Terpenuhi | Total data mencapai **110.577 baris** (setelah proses injeksi data). |
| **Minimal 12 Kolom** | Terpenuhi | Dataset utama memiliki total **14 kolom**. |
| **Minimal 2 Sumber Berbeda** | Terpenuhi | Menggunakan **Sumber 1** (Data Riwayat Janji Temu) dan **Sumber 2** (Data Wilayah). |
| **Minimal 2 Format Berbeda** | Terpenuhi | Sumber 1 memiliki format **CSV** dan Sumber 2 memiliki format **JSON**. |
| **Mengandung Missing Value** | Terpenuhi | Kolom `Age` memiliki nilai kosong (*null*) yang disimulasikan. |
| **Mengandung Duplicate Data** | Terpenuhi | Terdapat 50 baris duplikat (*duplicate rows*) untuk diuji pada tahap cleaning. |
| **Mengandung Outlier** | Terpenuhi | Terdapat nilai ekstrim pada kolom `Age` untuk pengujian deteksi anomali. |
| **Memiliki Kolom ID** | Terpenuhi | Menggunakan `AppointmentID` sebagai Primary Key dan `PatientId` sebagai Foreign Key. |

## **PIPELINE ETL**

### **EXTRACT**

In [ ]:
print("--- MENJALANKAN PIPELINE EXTRACT ---")

# Fungsi Pencatat Log
def log_process(nama_sumber, data, path_file, start_time):
    end_time = time.time()
    durasi = end_time - start_time
    ukuran = os.path.getsize(path_file) / 1024 # Hitung KB
    waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f" LOG: {nama_sumber}")
    print(f"   - Waktu Eksekusi : {waktu}")
    print(f"   - Durasi Eksekusi: {durasi:.4f} detik")
    print(f"   - Ukuran File    : {ukuran:.2f} KB")
    print(f"   - Dimensi Data   : {data.shape[0]} Baris, {data.shape[1]} Kolom")
    print("   ----------------------------------------")

# Fungsi Extract 1
def extract_etl_source1():
    print(" Sedang mengekstrak Source 1 (CSV)...")
    start_time = time.time()

    df = pd.read_csv('source1_appointments.csv')
    log_process("Kaggle Appointments (CSV)", df, 'source1_appointments.csv', start_time)

    # Simpan data mentah menggunakan variabel RAW_DIR dari Cell 0
    df.to_csv(f"{RAW_DIR}/source1_appointments_raw.csv", index=False)
    return df

# Fungsi Extract 2
def extract_etl_source2():
    print(" Sedang mengekstrak Source 2 (JSON)...")
    start_time = time.time()

    df = pd.read_json('source2_regions.json')
    log_process("Vitória Regions Mapping (JSON)", df, 'source2_regions.json', start_time)

    # Simpan data mentah menggunakan variabel RAW_DIR dari Cell 0
    df.to_json(f"{RAW_DIR}/source2_regions_raw.json", orient='records')
    return df

# Eksekusi
if os.path.exists('source1_appointments.csv') and os.path.exists('source2_regions.json'):
    df_utama = extract_etl_source1()
    df_pendukung = extract_etl_source2()
    print(f"\n[SUKSES] Tahap Extract Selesai! Data mentah tersimpan di: {RAW_DIR}")
else:
    print("ERROR: File sumber belum ada. Harap jalankan Cell 1 terlebih dahulu.")

--- MENJALANKAN PIPELINE EXTRACT ---
 Sedang mengekstrak Source 1 (CSV)...
 LOG: Kaggle Appointments (CSV)
   - Waktu Eksekusi : 2026-06-14 07:25:37
   - Durasi Eksekusi: 0.2407 detik
   - Ukuran File    : 10924.37 KB
   - Dimensi Data   : 110577 Baris, 14 Kolom
   ----------------------------------------
 Sedang mengekstrak Source 2 (JSON)...
 LOG: Vitória Regions Mapping (JSON)
   - Waktu Eksekusi : 2026-06-14 07:25:38
   - Durasi Eksekusi: 0.0047 detik
   - Ukuran File    : 4.20 KB
   - Dimensi Data   : 81 Baris, 2 Kolom
   ----------------------------------------

[SUKSES] Tahap Extract Selesai! Data mentah tersimpan di: /content/drive/MyDrive/Tugas_Besar_BigData_MaleakhiNymmo/raw


### **TRANSFORM**

#### Data Cleaning

Tujuan tahap cleaning adalah menghapus atau menghilangkan gangguan (noise) dan kesalahan pada data

    Data Duplikat: Menghapus baris data ganda berdasarkan Primary Key (AppointmentID).
    Strategi yang digunakan adalah keep='first', yaitu mempertahankan data pertama yang masuk dan membuang duplikat setelahnya.

    Penanganan Missing Values (Penghapusan/Drop): Ditemukan nilai kosong pada kolom Age.
    Strategi yang dipilih adalah melakukan penghapusan baris data secara langsung (dropna) pada entri yang kosong tersebut.

    Standarisasi Format Tanggal/Waktu: Mengonversi tipe data pada kolom ScheduledDay dan AppointmentDay dari format teks (string) menjadi format `datetime` standar.
    Standarisasi ini diimplementasikan agar data waktu valid untuk operasi perhitungan matematis pada tahap feature engineering.

    Penanganan Outlier (Filtering): Mendeteksi anomali pada kolom Age menggunakan batasan logika domain (domain constraint).
    Data dengan umur < 0 tahun (tidak mungkin) atau > 100 tahun (sangat jarang) diidentifikasi sebagai outlier.
    Setelah sistem memvalidasi bahwa jumlah outlier tersebut aman (di bawah batas toleransi 20%), data tersebut dihapus dari dataset untuk menjaga validitas analisis demografi


In [ ]:
print("--- [TRANSFORM: DATA CLEANING ---")

# 1. Load data mentah dari folder raw/ (menggunakan RAW_DIR dari Cell 0)
path_source1 = f"{RAW_DIR}/source1_appointments_raw.csv"
df_clean = pd.read_csv(path_source1)
baris_awal = len(df_clean)
print(f"Data awal dimuat: {baris_awal} baris.")

# 2. Menghapus Duplikat berdasarkan ID Pendaftaran (Primary Key)
dup_count = df_clean.duplicated(subset=['AppointmentID']).sum()
df_clean = df_clean.drop_duplicates(subset=['AppointmentID'], keep='first')
print(f"Telah Menghapus {dup_count} baris duplikat.")

# 3. Menangani Missing Value pada kolom Age
missing_count = df_clean['Age'].isna().sum()
df_clean = df_clean.dropna(subset=['Age'])
print(f"Telah Menghapus {missing_count} baris dengan Age kosong (Missing Value).")

# 4. Standarisasi Format Tanggal/Waktu
df_clean['ScheduledDay'] = pd.to_datetime(df_clean['ScheduledDay'])
df_clean['AppointmentDay'] = pd.to_datetime(df_clean['AppointmentDay'])
print(" Format tanggal pada 'ScheduledDay' dan 'AppointmentDay' telah distandarisasi.")

# 5. Deteksi dan Penanganan Outlier
lower_bound = 0
upper_bound = 100

# Cek kondisi outlier
kondisi_outlier = (df_clean['Age'] < lower_bound) | (df_clean['Age'] > upper_bound)
jumlah_outlier = kondisi_outlier.sum()
persentase_outlier = (jumlah_outlier / len(df_clean)) * 100

print(f"\nAnalisis Outlier Umur")
print(f"   - Batas Bawah: {lower_bound}")
print(f"   - Batas Atas : {upper_bound}")
print(f"   - Ditemukan  : {jumlah_outlier} baris outlier ({persentase_outlier:.2f}%)")

# Logika pengecekan 20% sesuai permintaan
if persentase_outlier > 20:
    print("Peringatan: Outlier lebih dari 20%. Disarankan menggunakan metode imputasi median.")
else:
    print(f"Outlier di bawah 20%. Menghapus baris outlier...")
    df_clean = df_clean[~kondisi_outlier]

baris_akhir = len(df_clean)
print(f"\nTahap Data Cleaning Selesai! Sisa data: {baris_akhir} baris.")

--- [TRANSFORM: DATA CLEANING ---
Data awal dimuat: 110577 baris.
Telah Menghapus 50 baris duplikat.
Telah Menghapus 51 baris dengan Age kosong (Missing Value).
 Format tanggal pada 'ScheduledDay' dan 'AppointmentDay' telah distandarisasi.

Analisis Outlier Umur
   - Batas Bawah: 0
   - Batas Atas : 100
   - Ditemukan  : 10 baris outlier (0.01%)
Outlier di bawah 20%. Menghapus baris outlier...

Tahap Data Cleaning Selesai! Sisa data: 110466 baris.


#### Standardisasi Data

Tujuan tahap standardisasi data adalah untuk menjaga konsistensi format di dalam Data Warehouse. Proses ini meliputi penyeragaman tipe data, penamaan, hingga transformasi nilai agar sesuai dengan standar analitik.

    Penyeragaman Nama Kolom: Mengubah seluruh nama kolom menjadi huruf kecil (lowercase) dan menerapkan format snake_case (contoh: dari PatientId menjadi patient_id).
    Hal ini meminimalisir error pemanggilan kolom (akibat case-sensitive) dan menjaga standar best practice penamaan di dalam database.

    Pembuatan Fitur Numerik Waktu: Memanipulasi kolom scheduled_day dan appointment_day (yang sudah terstandarisasi zona waktunya) untuk menciptakan fitur baru bernama waiting_days (selisih hari tunggu).
    Nilai divalidasi agar tidak ada yang negatif akibat potensi human-error pada saat pencatatan jadwal.

    Normalisasi Data Numerik: Menerapkan metode Min-Max Scaler pada dua kolom numerik yang memiliki rentang nilai berbeda, yaitu age dan waiting_days.
    Metode ini memampatkan skala nilai asli menjadi rentang standar 0.0 hingga 1.0 agar data lebih stabil dan tidak menghasilkan bias komputasi saat diolah lebih lanjut.

    Mengonversi data yang bernilai teks/kategorikal menjadi format numerik biner. Kolom gender diubah menjadi angka (F = 0, M = 1), dan variabel target is_no_show dikonversi (No/Datang = 0, Yes/Tidak Datang = 1)
    agar siap digunakan dalam kalkulasi rasio atau pemodelan analitik.

    Konsistensi Tipe Data: Melakukan casting secara eksplisit untuk memastikan integritas data.
    Kolom identitas seperti patient_id dan appointment_id dikunci menjadi bilangan bulat 64-bit (int64), serta kolom age dipastikan dalam format int tanpa desimal sebelum dimuat ke Data Warehouse.


In [ ]:
print("--- TRANSFORM: STANDARDISASI DATA ---")

# 1. Menyeragamkan penamaan kolom (lowercase dan snake_case)
df_clean.columns = df_clean.columns.str.lower()
df_clean = df_clean.rename(columns={
    'patientid': 'patient_id',
    'appointmentid': 'appointment_id',
    'scheduledday': 'scheduled_day',
    'appointmentday': 'appointment_day',
    'neighbourhood': 'neighbourhood_name',
    'no-show': 'is_no_show'
})
print("Penamaan kolom telah diseragamkan menjadi snake_case.")

# Pengaman ganda: Pastikan kolom sudah bertipe datetime dan zona waktunya diseragamkan (UTC)
df_clean['scheduled_day'] = pd.to_datetime(df_clean['scheduled_day'], utc=True)
df_clean['appointment_day'] = pd.to_datetime(df_clean['appointment_day'], utc=True)

# Membuat 1 fitur numerik tambahan (waiting_days)
# Gunakan .dt.normalize() untuk membuang jam/menit dengan aman
df_clean['waiting_days'] = (df_clean['appointment_day'].dt.normalize() - df_clean['scheduled_day'].dt.normalize()).dt.days

# Pastikan tidak ada nilai negatif (jika jadwal mundur karena kesalahan input)
df_clean['waiting_days'] = df_clean['waiting_days'].apply(lambda x: max(x, 0))

# 2. Normalisasi minimal 2 kolom numerik (age dan waiting_days)
scaler = MinMaxScaler()
df_clean[['age_normalized', 'waiting_days_normalized']] = scaler.fit_transform(df_clean[['age', 'waiting_days']])
print("Normalisasi (Min-Max Scaler) diterapkan pada kolom 'age' dan 'waiting_days'.")

# 3. Encoding pada kolom kategorikal (gender dan is_no_show)
# Gender: F = 0, M = 1
df_clean['gender'] = df_clean['gender'].map({'F': 0, 'M': 1})
# is_no_show: No (Datang) = 0, Yes (Tidak Datang) = 1
df_clean['is_no_show'] = df_clean['is_no_show'].map({'No': 0, 'Yes': 1})
print("Encoding selesai (Gender: M/F -> 1/0, is_no_show: Yes/No -> 1/0).")

# 4. Konsistensi Tipe Data
df_clean['patient_id'] = df_clean['patient_id'].astype('int64')
df_clean['appointment_id'] = df_clean['appointment_id'].astype('int64')
df_clean['age'] = df_clean['age'].astype('int')
print("Konsistensi tipe data telah divalidasi.")

print("\nData Setelah Standardisasi:")
display(df_clean[['patient_id', 'age', 'age_normalized', 'gender', 'is_no_show', 'waiting_days', 'waiting_days_normalized']].head(3))

--- TRANSFORM: STANDARDISASI DATA ---
Penamaan kolom telah diseragamkan menjadi snake_case.
Normalisasi (Min-Max Scaler) diterapkan pada kolom 'age' dan 'waiting_days'.
Encoding selesai (Gender: M/F -> 1/0, is_no_show: Yes/No -> 1/0).
Konsistensi tipe data telah divalidasi.

Data Setelah Standardisasi:


,patient_id,age,age_normalized,gender,is_no_show,waiting_days,waiting_days_normalized
2,4262962299951,62,0.62,0,0,0,0.0
3,867951213174,8,0.08,0,0,0,0.0
4,8841186448183,56,0.56,0,0,0,0.0


#### Data Enrichment dan Feature Engineering

Tujuan tahap data enrichment & feature engineering adalah untuk meningkatkan nilai analitik dari dataset dengan cara menggabungkan informasi dari sumber eksternal serta menciptakan variabel-variabel baru yang lebih bermakna untuk kebutuhan visualisasi dan pencarian insight.

    Data Enrichment (Penggabungan Data): Melakukan integrasi (Left Join) antara tabel utama pendaftaran medis dengan tabel dimensi wilayah (Source 2/JSON).
    Penggabungan menggunakan kunci lokasi (neighbourhood_name = Lokasi) agar setiap baris pendaftaran pasien memiliki konteks geografis wilayah administratif resmi (region_name).

    Feature Engineering (Pembuatan Fitur Baru): Mengekstraksi dan mentransformasi kolom yang sudah ada untuk menghasilkan 5 fitur turunan baru yang dirancang khusus untuk menganalisis pola ketidakhadiran pasien:
    - waiting_days: (Diinisiasi pada tahap standardisasi) Menyimpan perhitungan selisih hari antara tanggal pasien mendaftar dengan tanggal janji temu dilaksanakan.
    - is_same_day: Indikator biner yang secara spesifik mengidentifikasi pasien walk-in atau mereka yang datang di hari yang sama saat mendaftar (bernilai 1 jika waiting_days bernilai 0).
    - is_weekend: Indikator biner turunan dari tanggal janji temu untuk mengidentifikasi apakah jadwal jatuh pada akhir pekan (Sabtu/Minggu bernilai 1) atau hari kerja biasa (bernilai 0).-
    - age_group: Mengubah data numerik umur menjadi data kategorikal (Anak, Remaja, Dewasa, Lansia) menggunakan binning.
    - day_of_week: Mengekstrak hari operasional dalam seminggu (Senin = 0, hingga Minggu = 6) untuk melihat pola kepadatan jadwal harian rumah sakit secara historis.
    - total_comorbidities: Fitur agregasi yang menjumlahkan 4 kondisi kesehatan bawaan pasien (hipertensi, diabetes, alkoholisme, dan disabilitas).

In [ ]:
print("--- TRANSFORM: DATA ENRICHMENT & FEATURE ENGINEERING ---")

# ==========================================
# 1. DATA ENRICHMENT (Join 2 Sumber Data)
# ==========================================
# Load Source 2 (JSON Wilayah) dari direktori raw
path_source2 = f"{RAW_DIR}/source2_regions_raw.json"
df_regions = pd.read_json(path_source2)

# Lakukan Left Join:
# Data pendaftaran (df_clean) disambungkan ke data wilayah (df_regions)
df_enriched = pd.merge(df_clean, df_regions, left_on='neighbourhood_name', right_on='Lokasi', how='left')

# Rapikan tabel setelah join
df_enriched = df_enriched.drop(columns=['Lokasi']) # Hapus karena redundan
df_enriched = df_enriched.rename(columns={'Region': 'region_name'})

# ==========================================
# 2. FEATURE ENGINEERING (5 Fitur Baru)
# ==========================================
# Kita rename lead_time_days dari Cell 4 menjadi waiting_days agar sesuai konsepmu
df_enriched = df_enriched.rename(columns={
    'lead_time_days': 'waiting_days',
    'lead_time_normalized': 'waiting_days_normalized'
})

# Fitur 1: waiting_days
# (Telah tersedia dari tahap sebelumnya hasil rename di atas)

# Fitur 2: is_same_day (Indikator biner datang di hari yang sama)
df_enriched['is_same_day'] = (df_enriched['waiting_days'] == 0).astype(int)

# Fitur 3: is_weekend (1 jika Sabtu/Minggu, 0 jika hari kerja)
df_enriched['is_weekend'] = df_enriched['appointment_day'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# Fitur 4: age_group (Kategorisasi Umur: Anak, Remaja, Dewasa, Lansia)
bins = [-1, 12, 19, 59, 150] # -1 untuk bayi, batas atas lansia 150
labels = ['Anak', 'Remaja', 'Dewasa', 'Lansia']
df_enriched['age_group'] = pd.cut(df_enriched['age'], bins=bins, labels=labels)

# Fitur 5: day_of_week (Senin=0, Minggu=6)
df_enriched['day_of_week'] = df_enriched['appointment_day'].dt.dayofweek

# Fitur 6: total_comorbidities (Total penyakit bawaan)
df_enriched['total_comorbidities'] = df_enriched['hipertension'] + df_enriched['diabetes'] + df_enriched['alcoholism'] + df_enriched['handcap']

print("Data Enrichment (Merge) Selesai.")
print("Feature Engineering Selesai. 6 fitur baru berhasil ditambahkan.")

# Pratinjau hasil penggabungan dan pembuatan fitur baru
fitur_baru = ['neighbourhood_name', 'region_name', 'waiting_days', 'is_same_day', 'is_weekend', 'age_group', 'day_of_week', 'total_comorbidities']
print("\nPratinjau Hasil Enrichment & Fitur Baru:")
display(df_enriched[fitur_baru].head())

--- TRANSFORM: DATA ENRICHMENT & FEATURE ENGINEERING ---
Data Enrichment (Merge) Selesai.
Feature Engineering Selesai. 6 fitur baru berhasil ditambahkan.

Pratinjau Hasil Enrichment & Fitur Baru:


,neighbourhood_name,region_name,waiting_days,is_same_day,is_weekend,age_group,day_of_week,total_comorbidities
0,MATA DA PRAIA,Região 9,0,1,0,Lansia,4,0
1,PONTAL DE CAMBURI,Região 9,0,1,0,Anak,4,0
2,JARDIM DA PENHA,Região 9,0,1,0,Dewasa,4,2
3,REPÚBLICA,Região 9,2,0,0,Lansia,4,1
4,GOIABEIRAS,Região 6,2,0,0,Dewasa,4,0


#### Validasi Kualitas Data

Tujuan tahap validasi kualitas data adalah untuk mengevaluasi dan memastikan bahwa dataset yang telah melalui proses pembersihan dan transformasi memiliki integritas yang tinggi sebelum dimuat ke dalam Data Warehouse. Proses validasi ini mencakup 6 aturan pengecekan otomatis:

    Uniqueness Check: Memastikan tidak ada duplikasi pada Primary Key (appointment_id). Validasi ini menjamin bahwa setiap baris di dalam tabel merepresentasikan satu catatan pendaftaran yang unik dan valid.

    Null Check: Memverifikasi tingkat kelengkapan data dengan memastikan tidak ada nilai kosong (missing value) yang tersisa pada kolom-kolom fundamental, seperti identitas pasien (patient_id), ID pendaftaran (appointment_id), dan age.

    Range Check: Menguji validitas logika batasan nilai (domain batas) pada kolom numerik. Aturan ini memastikan age berada pada rentang yang masuk akal (-1 untuk bayi baru lahir hingga 150 tahun) dan waiting_days tidak memiliki nilai negatif.

    Datatype Consistency: Memvalidasi bahwa arsitektur tipe data sudah konsisten dan siap diproses oleh database, yaitu memastikan appointment_id tersimpan sebagai integer (bilangan bulat) dan age dikenali secara matematis sebagai format numerik.

    Referential Integrity Check: Mengevaluasi integritas relasi data setelah proses Data Enrichment. Validasi ini memastikan semua data kecamatan awal berhasil dipetakan ke dimensi wilayah administratif (region_name) tanpa ada baris yang gagal terhubung (bernilai NaN).

    Distribusi Data Check: Melakukan pengecekan proporsi informasi pada kolom target is_no_show. Langkah ini menghitung persentase rasio keseimbangan kelas antara pasien yang datang (0) berbanding pasien yang tidak datang (1)

In [ ]:
print("--- TRANSFORM: VALIDASI KUALITAS DATA ---")

# Siapkan list untuk mencatat status validasi
validation_logs = []

# 1. Uniqueness Check (Apakah ID Pendaftaran unik?)
is_unique = df_enriched['appointment_id'].is_unique
if is_unique:
    validation_logs.append("[LULUS] Uniqueness Check: Tidak ada duplikasi pada 'appointment_id'.")
else:
    validation_logs.append("[GAGAL] Uniqueness Check: Ditemukan duplikasi pada 'appointment_id'.")

# 2. Null Check (Apakah ada missing value tersisa di kolom penting?)
null_count = df_enriched[['patient_id', 'appointment_id', 'age']].isnull().sum().sum()
if null_count == 0:
    validation_logs.append("[LULUS] Null Check: Tidak ada missing value pada kolom utama.")
else:
    validation_logs.append(f"[GAGAL] Null Check: Terdapat {null_count} missing value.")
    # Perbaikan jika gagal: df_enriched = df_enriched.dropna(subset=['patient_id', 'appointment_id', 'age'])

# 3. Range Check (Apakah umur dan waiting_days masuk akal?)
# Umur harus antara -1 (bayi baru lahir di data Kaggle) sampai 115, waiting_days tidak boleh negatif
range_valid = df_enriched['age'].between(-1, 150).all() and df_enriched['waiting_days'].ge(0).all()
if range_valid:
    validation_logs.append("[LULUS] Range Check: Umur dan waiting_days berada dalam batas normal.")
else:
    validation_logs.append("[GAGAL] Range Check: Ditemukan nilai di luar batas normal.")

# 4. Datatype Consistency (Apakah tipe data konsisten?)
type_valid = pd.api.types.is_integer_dtype(df_enriched['appointment_id']) and pd.api.types.is_numeric_dtype(df_enriched['age'])
if type_valid:
    validation_logs.append("[LULUS] Datatype Consistency: Tipe data ID (integer) dan umur (numerik) konsisten.")
else:
    validation_logs.append("[GAGAL] Datatype Consistency: Ada tipe data yang tidak sesuai format.")

# 5. Referential Integrity Check (Apakah hasil Join Wilayah memiliki nilai kosong?)
# Memastikan semua neighbourhood dari Kaggle berhasil dipetakan ke region resmi
unmapped_regions = df_enriched['region_name'].isnull().sum()
if unmapped_regions == 0:
    validation_logs.append("[LULUS] Referential Integrity: Semua kecamatan (neighbourhood) berhasil dipetakan ke Region.")
else:
    validation_logs.append(f"[GAGAL] Referential Integrity: Ada {unmapped_regions} kecamatan yang gagal dipetakan (Nilai NaN).")

# 6. Distribusi Data Check (Melihat keseimbangan data target is_no_show)
distribusi_target = df_enriched['is_no_show'].value_counts(normalize=True) * 100
validation_logs.append(f"[INFO] Distribusi Data 'is_no_show': \n    - Datang (0)      : {distribusi_target[0]:.2f}%\n    - Tidak Datang (1): {distribusi_target.get(1, 0):.2f}%")

# Tampilkan Log Hasil Validasi
for log in validation_logs:
    print(log)



--- TRANSFORM: VALIDASI KUALITAS DATA ---
[LULUS] Uniqueness Check: Tidak ada duplikasi pada 'appointment_id'.
[LULUS] Null Check: Tidak ada missing value pada kolom utama.
[LULUS] Range Check: Umur dan waiting_days berada dalam batas normal.
[LULUS] Datatype Consistency: Tipe data ID (integer) dan umur (numerik) konsisten.
[LULUS] Referential Integrity: Semua kecamatan (neighbourhood) berhasil dipetakan ke Region.
[INFO] Distribusi Data 'is_no_show': 
    - Datang (0)      : 79.81%
    - Tidak Datang (1): 20.19%


## **LOAD**

Tujuan tahap Load adalah untuk menyimpan secara permanen data yang telah dibersihkan, distandarisasi, dan diperkaya ke dalam sistem Data Warehouse tujuan, sehingga data tersebut siap diakses untuk kebutuhan analisis dan visualisasi akhir.

    Setup Direktori & Backup Fisik (CSV): Membangun struktur direktori warehouse/ secara otomatis untuk mengisolasi hasil akhir pipeline.
    Sistem kemudian mengekspor salinan data final ke dalam format CSV (fact_appointments.csv).

    Integrasi Serverless Database (SQLite): Menginisiasi koneksi ke sistem database SQLite (warehouse_appointment.db).
    Pendekatan serverless ini dipilih karena sangat efisien, stabil dijalankan di dalam ekosistem cloud notebook (Google Colab), dan tidak terhambat oleh isu keamanan jaringan (firewall/localhost porting).

    Pemuatan Data ke Tabel SQL (Ingestion): Mentransfer seluruh data dari memory (DataFrame Pandas) secara langsung ke dalam tabel relasional bernama fact_appointments.
    Proses ini menggunakan strategi operasi if_exists='replace' untuk memastikan bahwa setiap kali pipeline ETL dijalankan, tabel selalu diperbarui dengan versi data yang paling mutakhir tanpa risiko duplikasi penumpukan data lama.

    Verifikasi Integritas & Keamanan Koneksi: Setelah data dimuat, sistem secara otomatis menjalankan query SQL dasar (SELECT COUNT(*)) untuk memvalidasi bahwa jumlah baris yang terekam di dalam database sudah sesuai dengan ukuran data final.
    Di akhir proses, sistem menerapkan protokol keamanan (safe close) untuk menutup koneksi database agar file tidak mengalami kerusakan (corrupt) dan memori komputasi dapat dibebaskan.

In [ ]:
import os
import sqlite3
import pandas as pd
import time
from datetime import datetime

print("--- [LANGKAH 4A] ETL LOAD: WAREHOUSE DENGAN STAR SCHEMA ---")

# ==========================================
# 0. Setup Direktori & Fungsi Logging
# ==========================================
WAREHOUSE_DIR = f"{PROJECT_ROOT}/warehouse" if 'PROJECT_ROOT' in locals() else "warehouse"
os.makedirs(WAREHOUSE_DIR, exist_ok=True)
db_path = f"{WAREHOUSE_DIR}/warehouse_appointment.db"

def log_load(tabel, status, start_t):
    end_t = time.time()
    waktu = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{waktu}] {status} tabel '{tabel}' (Durasi: {end_t - start_t:.4f} detik)")

# ==========================================
# 1. Pembuatan Star Schema (1 Fakta, 2 Dimensi)
# ==========================================
print("\n1. Mempersiapkan Data untuk Star Schema...")

# [PERBAIKAN 1]: Paksa patient_id menjadi string (teks) di dataframe awal
# agar angka panjangnya tidak berubah jadi notasi ilmiah (E+13)
df_enriched['patient_id'] = df_enriched['patient_id'].astype(str)

# A. Dimensi Pasien (dim_patient)
dim_patient = df_enriched[['patient_id', 'age', 'age_group', 'gender', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'total_comorbidities']].drop_duplicates(subset=['patient_id'])

# B. Dimensi Lokasi (dim_location)
dim_location = df_enriched[['neighbourhood_name', 'region_name']].drop_duplicates().reset_index(drop=True)
dim_location['location_id'] = dim_location.index + 1

# C. Tabel Fakta (fact_appointments)
df_fact = pd.merge(df_enriched, dim_location, on=['neighbourhood_name', 'region_name'], how='left')
fact_appointments = df_fact[['appointment_id', 'patient_id', 'location_id', 'scheduled_day', 'appointment_day', 'waiting_days', 'waiting_days_normalized', 'is_same_day', 'is_weekend', 'day_of_week', 'is_no_show']]

print("Pemecahan data ke dalam 1 Tabel Fakta dan 2 Tabel Dimensi selesai.")

# ==========================================
# 2. Membuat Skema Database (DDL dengan PK & FK) & Memuat Data
# ==========================================
print("\n2. Membangun Skema dan Memuat Data ke SQLite...")
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Aktifkan constraint Foreign Key di SQLite
    cursor.execute("PRAGMA foreign_keys = ON;")

    # Drop tabel jika sudah ada agar skema ter-reset bersih
    cursor.execute("DROP TABLE IF EXISTS fact_appointments;")
    cursor.execute("DROP TABLE IF EXISTS dim_location;")
    cursor.execute("DROP TABLE IF EXISTS dim_patient;")

    # DDL: Tabel Dimensi Pasien
    cursor.execute('''
        CREATE TABLE dim_patient (
            patient_id TEXT PRIMARY KEY,
            age INTEGER,
            age_group TEXT,
            gender INTEGER,
            hipertension INTEGER,
            diabetes INTEGER,
            alcoholism INTEGER,
            handcap INTEGER,
            total_comorbidities INTEGER
        )
    ''')

    # DDL: Tabel Dimensi Lokasi
    cursor.execute('''
        CREATE TABLE dim_location (
            location_id INTEGER PRIMARY KEY,
            neighbourhood_name TEXT,
            region_name TEXT
        )
    ''')

    # DDL: Tabel Fakta
    cursor.execute('''
        CREATE TABLE fact_appointments (
            appointment_id INTEGER PRIMARY KEY,
            patient_id TEXT,  -- [PERBAIKAN 2]: Ubah dari INTEGER menjadi TEXT
            location_id INTEGER,
            scheduled_day TEXT,
            appointment_day TEXT,
            waiting_days INTEGER,
            waiting_days_normalized REAL,
            is_same_day INTEGER,
            is_weekend INTEGER,
            day_of_week INTEGER,
            is_no_show INTEGER,
            FOREIGN KEY (patient_id) REFERENCES dim_patient (patient_id),
            FOREIGN KEY (location_id) REFERENCES dim_location (location_id)
        )
    ''')

    # Load Data
    start_time = time.time()
    dim_patient.to_sql('dim_patient', conn, if_exists='append', index=False)
    log_load("dim_patient", "Berhasil memuat", start_time)

    start_time = time.time()
    dim_location.to_sql('dim_location', conn, if_exists='append', index=False)
    log_load("dim_location", "Berhasil memuat", start_time)

    start_time = time.time()
    fact_appointments.to_sql('fact_appointments', conn, if_exists='append', index=False)
    log_load("fact_appointments", "Berhasil memuat", start_time)

except Exception as e:
    print(f"Error saat membangun Skema/Load: {e}")
finally:
    if 'conn' in locals():
        conn.close()
        print("\nKoneksi database ditutup. Tahap (Load & Star Schema) Selesai Sempurna!")

--- [LANGKAH 4A] ETL LOAD: WAREHOUSE DENGAN STAR SCHEMA ---

1. Mempersiapkan Data untuk Star Schema...
Pemecahan data ke dalam 1 Tabel Fakta dan 2 Tabel Dimensi selesai.

2. Membangun Skema dan Memuat Data ke SQLite...
[2026-06-14 07:25:45] Berhasil memuat tabel 'dim_patient' (Durasi: 0.5695 detik)
[2026-06-14 07:25:45] Berhasil memuat tabel 'dim_location' (Durasi: 0.0190 detik)
[2026-06-14 07:26:16] Berhasil memuat tabel 'fact_appointments' (Durasi: 31.0083 detik)

Koneksi database ditutup. Tahap (Load & Star Schema) Selesai Sempurna!


### Melihat isi dari tabel SQLite

In [ ]:
import sqlite3
import pandas as pd

print("--- MENAMPILKAN DATA DARI SQLITE ---")

# Gunakan variabel db_path dari tahap Load sebelumnya
db_path = f"{WAREHOUSE_DIR}/warehouse_appointment.db"

try:
    print("Menghubungkan ke database SQLite...")
    koneksi_sqlite = sqlite3.connect(db_path)

    # Menulis query SQL asli untuk mengambil 5 baris pertama
    query_sql = "SELECT * FROM fact_appointments LIMIT 5;"

    # Eksekusi query dan muat langsung ke dalam DataFrame Pandas
    df_dari_db = pd.read_sql_query(query_sql, koneksi_sqlite)

    print("Berhasil menarik data menggunakan SQL Query!")
    print("\nTabel 'fact_appointments' dari Database:\n")
    display(df_dari_db)

    # Melihat isi dimensi pasien
    print("\nTabel 'dim_patient' dari Database:\n")
    query_cek_pasien = "SELECT * FROM dim_patient LIMIT 5;"

    display(pd.read_sql_query(query_cek_pasien, koneksi_sqlite))

    # Melihat isi dim lokasi
    print("\nTabel 'dim_location' dari Database:\n")
    query_cek_lokasi = "SELECT * FROM dim_location LIMIT 5;"

    display(pd.read_sql_query(query_cek_lokasi, koneksi_sqlite))

except Exception as e:
    print(f"[ERROR] Gagal membaca data: {e}")

finally:
    # Selalu pastikan koneksi ditutup setelah selesai melihat data
    if 'koneksi_sqlite' in locals():
        koneksi_sqlite.close()
        print("\nKoneksi database ditutup dengan aman.")

--- MENAMPILKAN DATA DARI SQLITE ---
Menghubungkan ke database SQLite...
Berhasil menarik data menggunakan SQL Query!

Tabel 'fact_appointments' dari Database:



,appointment_id,patient_id,location_id,scheduled_day,appointment_day,waiting_days,waiting_days_normalized,is_same_day,is_weekend,day_of_week,is_no_show
0,5030230,832256398961987,26,2015-11-10 07:13:56+00:00,2016-05-04 00:00:00+00:00,176,0.983240,0,0,2,0
1,5122866,91637474953513,66,2015-12-03 08:17:28+00:00,2016-05-02 00:00:00+00:00,151,0.843575,0,0,0,1
2,5134197,1216586867796,14,2015-12-07 10:40:59+00:00,2016-06-03 00:00:00+00:00,179,1.000000,0,0,4,1
3,5134220,31899595421534,15,2015-12-07 10:42:42+00:00,2016-06-03 00:00:00+00:00,179,1.000000,0,0,4,0
4,5134223,9582232334148,14,2015-12-07 10:43:01+00:00,2016-06-03 00:00:00+00:00,179,1.000000,0,0,4,0



Tabel 'dim_patient' dari Database:



,patient_id,age,age_group,gender,hipertension,diabetes,alcoholism,handcap,total_comorbidities
0,4262962299951,62,Lansia,0,0,0,0,0,0
1,867951213174,8,Anak,0,0,0,0,0,0
2,8841186448183,56,Dewasa,0,1,1,0,0,2
3,95985133231274,76,Lansia,0,1,0,0,0,1
4,733688164476661,23,Dewasa,0,0,0,0,0,0



Tabel 'dim_location' dari Database:



,location_id,neighbourhood_name,region_name
0,1,MATA DA PRAIA,Região 9
1,2,PONTAL DE CAMBURI,Região 9
2,3,JARDIM DA PENHA,Região 9
3,4,REPÚBLICA,Região 9
4,5,GOIABEIRAS,Região 6



Koneksi database ditutup dengan aman.


### Query Analitik

In [ ]:
import sqlite3
import pandas as pd
import os

print("=== [LANGKAH 4B] EKSPLORASI: 8 QUERY ANALITIK WAJIB ===")

# Pastikan path sesuai dengan nama file database-mu
WAREHOUSE_DIR = f"{PROJECT_ROOT}/warehouse" if 'PROJECT_ROOT' in locals() else "warehouse"
db_path = f"{WAREHOUSE_DIR}/warehouse_appointment.db"

# Buka Koneksi ke Database
koneksi_sqlite = sqlite3.connect(db_path)

def run_query(judul, sql_query):
    print(f"\n🔹 {judul}")
    try:
        # Baca hasil SQL langsung ke DataFrame biar rapi
        df_hasil = pd.read_sql(sql_query, koneksi_sqlite)
        # Gunakan display agar tabel berbentuk rapi di Colab
        display(df_hasil)
    except Exception as e:
        print(f"❌ Error: {e}")

# --- QUERY 1: Validasi Total Data ---
q1 = "SELECT COUNT(*) as total_antrean FROM fact_appointments"
run_query("1. Total Transaksi dalam Warehouse", q1)

# --- QUERY 2: Menghitung Rasio No-Show (is_no_show 1 = Tidak Datang) ---
q2 = """
SELECT
    CASE WHEN is_no_show = 1 THEN 'Tidak Hadir' ELSE 'Hadir' END as status,
    COUNT(*) as jumlah,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_appointments), 2) as persentase_persen
FROM fact_appointments
GROUP BY is_no_show
"""
run_query("2. Rasio Kehadiran Pasien (Show vs No-Show)", q2)

# --- QUERY 3: Top 5 Lokasi dengan Pasien Terbanyak ---
# Menggunakan JOIN ke dim_location karena lokasi ada di tabel dimensi
q3 = """
SELECT l.neighbourhood_name, COUNT(f.appointment_id) as total_pasien
FROM fact_appointments f
JOIN dim_location l ON f.location_id = l.location_id
GROUP BY l.neighbourhood_name
ORDER BY total_pasien DESC
LIMIT 5
"""
run_query("3. Top 5 Lokasi (Neighbourhood) Tersibuk", q3)

# --- QUERY 4: Analisis Berdasarkan Gender ---
# Menggunakan JOIN ke dim_patient. (Gender: 0=F, 1=M)
q4 = """
SELECT
    CASE WHEN p.gender = 1 THEN 'Laki-laki (M)' ELSE 'Perempuan (F)' END as gender,
    COUNT(f.appointment_id) as total_janji,
    SUM(f.is_no_show) as total_tidak_hadir,
    ROUND(AVG(f.is_no_show) * 100, 2) as persen_tidak_hadir
FROM fact_appointments f
JOIN dim_patient p ON f.patient_id = p.patient_id
GROUP BY p.gender
"""
run_query("4. Statistik Ketidakhadiran berdasarkan Gender", q4)

# --- QUERY 5: Rata-rata Waktu Tunggu (Waiting Days) ---
q5 = """
SELECT
    CASE WHEN is_no_show = 1 THEN 'Tidak Hadir' ELSE 'Hadir' END as status,
    ROUND(AVG(waiting_days), 1) as rata_rata_hari_tunggu
FROM fact_appointments
GROUP BY is_no_show
"""
run_query("5. Rata-rata Waktu Tunggu (Hadir vs Tidak)", q5)

# --- QUERY 6: Analisis Akhir Pekan (Weekend Effect) ---
q6 = """
SELECT
    CASE WHEN is_weekend = 1 THEN 'Akhir Pekan' ELSE 'Hari Kerja' END as tipe_hari,
    COUNT(*) as total,
    ROUND(AVG(is_no_show) * 100, 2) as persen_tidak_hadir
FROM fact_appointments
GROUP BY is_weekend
"""
run_query("6. Tren Ketidakhadiran: Weekend vs Weekday", q6)

# --- QUERY 7: Distribusi per Region ---
# Join dengan Dimensi Lokasi
q7 = """
SELECT
    l.region_name as region,
    COUNT(f.appointment_id) as jumlah_pasien
FROM fact_appointments f
JOIN dim_location l ON f.location_id = l.location_id
GROUP BY l.region_name
ORDER BY jumlah_pasien DESC
"""
run_query("7. Sebaran Pasien per Wilayah Kesehatan", q7)

# --- QUERY 8: Analisis Kategori Umur ---
# Menggunakan kolom age_group yang sudah kita buat di tabel dim_patient (Star Schema power!)
q8 = """
SELECT
    p.age_group as kategori_umur,
    COUNT(f.appointment_id) as total_pasien,
    SUM(f.is_no_show) as tidak_hadir,
    ROUND(AVG(f.is_no_show) * 100, 2) as persen_no_show
FROM fact_appointments f
JOIN dim_patient p ON f.patient_id = p.patient_id
GROUP BY p.age_group
ORDER BY persen_no_show DESC
"""
run_query("8. Kelompok Umur Paling Sering Mangkir", q8)

# Tutup koneksi setelah selesai
koneksi_sqlite.close()
print("\n✅ [SELESAI] Seluruh 8 Query Analitik Berhasil Dijalankan dan Koneksi Ditutup.")

=== [LANGKAH 4B] EKSPLORASI: 8 QUERY ANALITIK WAJIB ===

🔹 1. Total Transaksi dalam Warehouse


,total_antrean
0,110466



🔹 2. Rasio Kehadiran Pasien (Show vs No-Show)


,status,jumlah,persentase_persen
0,Hadir,88164,79.81
1,Tidak Hadir,22302,20.19



🔹 3. Top 5 Lokasi (Neighbourhood) Tersibuk


...


🔹 7. Sebaran Pasien per Wilayah Kesehatan


,region,jumlah_pasien
0,Região 4,23913
1,Região 7,18018
2,Região 3,16882
3,Região 2,13405
4,Região 6,9904
5,Região 1,9306
6,Região 8,7718
7,Região 9,5831
8,Região 5,4060
9,Outros/Luar Vitória,1429



🔹 8. Kelompok Umur Paling Sering Mangkir


,kategori_umur,total_pasien,tidak_hadir,persen_no_show
0,Remaja,9357,2429,25.96
1,Dewasa,58912,12329,20.93
2,Anak,21048,4310,20.48
3,Lansia,21149,3234,15.29



✅ [SELESAI] Seluruh 8 Query Analitik Berhasil Dijalankan dan Koneksi Ditutup.


## **PIPELINE ELT**

### Extract dan Load

In [ ]:
import pandas as pd
import sqlite3
import os

print("--- [ELT - 7.1] EXTRACT & LOAD (DATA LAKE) ---")

# 1. Setup Data Lake (Database baru khusus untuk raw data)
DATA_LAKE_DIR = f"{PROJECT_ROOT}/data_lake"
os.makedirs(DATA_LAKE_DIR, exist_ok=True)
db_lake_path = f"{DATA_LAKE_DIR}/datalake_appointment.db"

# 2. EXTRACT: Mengambil dataset mentah (TANPA preprocessing)
# Menggunakan file mentah yang kita buat di awal banget (yang sengaja dikotorin)
raw_appointments = pd.read_csv('source1_appointments.csv')
raw_regions = pd.read_json('source2_regions.json')

# 3. LOAD: Memuat data langsung ke dalam tabel raw di Data Lake
try:
    conn_lake = sqlite3.connect(db_lake_path)

    # Load ke tabel raw
    raw_appointments.to_sql('raw_appointments', conn_lake, index=False, if_exists='replace')
    raw_regions.to_sql('raw_regions', conn_lake, index=False, if_exists='replace')

    # 4. METADATA: Dokumentasi proses load sesuai syarat rubrik
    print("✅ Berhasil memuat data mentah ke Data Lake (SQLite).")
    print("\n📊 METADATA PROSES LOAD:")

    # Metadata Source 1
    size_src1 = os.path.getsize('source1_appointments.csv') / (1024 * 1024)
    print(f"   [Tabel: raw_appointments]")
    print(f"   - Ukuran File : {size_src1:.2f} MB")
    print(f"   - Total Baris : {len(raw_appointments)}")
    print(f"   - Struktur/Kolom: {', '.join(raw_appointments.columns)}")

    # Metadata Source 2
    size_src2 = os.path.getsize('source2_regions.json') / 1024
    print(f"\n   [Tabel: raw_regions]")
    print(f"   - Ukuran File : {size_src2:.2f} KB")
    print(f"   - Total Baris : {len(raw_regions)}")
    print(f"   - Struktur/Kolom: {', '.join(raw_regions.columns)}")

except Exception as e:
    print(f"⚠️ Error saat Load ke Data Lake: {e}")
finally:
    if 'conn_lake' in locals():
        conn_lake.close()

--- [ELT - 7.1] EXTRACT & LOAD (DATA LAKE) ---
✅ Berhasil memuat data mentah ke Data Lake (SQLite).

📊 METADATA PROSES LOAD:
   [Tabel: raw_appointments]
   - Ukuran File : 10.67 MB
   - Total Baris : 110577
   - Struktur/Kolom: PatientId, AppointmentID, Gender, ScheduledDay, AppointmentDay, Age, Neighbourhood, Scholarship, Hipertension, Diabetes, Alcoholism, Handcap, SMS_received, No-show

   [Tabel: raw_regions]
   - Ukuran File : 4.20 KB
   - Total Baris : 81
   - Struktur/Kolom: Lokasi, Region


### Load

In [ ]:
import sqlite3
import pandas as pd

print("--- [ELT - 7.2] TRANSFORMASI IN-WAREHOUSE (SQL) ---")

db_lake_path = f"{DATA_LAKE_DIR}/datalake_appointment.db"
conn_lake = sqlite3.connect(db_lake_path)
cursor = conn_lake.cursor()

# =======================================================
# TRANSFORMASI 1, 2 & 4: Cleaning, Join, dan Feature Engineering via SQL
# Kita buat tabel baru (analytics_appointments) langsung di dalam database
# =======================================================
sql_transform = """
CREATE TABLE IF NOT EXISTS analytics_appointments AS
WITH CleanedData AS (
    -- Subquery untuk Cleaning Data (Menghapus Duplikat & Filter Outlier/Null)
    SELECT
        PatientId,
        AppointmentID,
        Gender,
        ScheduledDay,
        AppointmentDay,
        Age,
        Neighbourhood,
        Hipertension,
        Diabetes,
        Alcoholism,
        Handcap,
        SMS_received,
        "No-show" AS is_no_show
    FROM raw_appointments
    WHERE Age IS NOT NULL
      AND Age BETWEEN 0 AND 100
    GROUP BY AppointmentID -- Trik SQL untuk hapus duplikat berdasarkan Primary Key
)
SELECT
    c.PatientId,
    c.AppointmentID,
    c.Gender,
    c.Age,
    -- FEATURE ENGINEERING 1: Kategorisasi Umur menggunakan CASE WHEN
    CASE
        WHEN c.Age <= 12 THEN 'Anak'
        WHEN c.Age <= 19 THEN 'Remaja'
        WHEN c.Age <= 59 THEN 'Dewasa'
        ELSE 'Lansia'
    END AS age_group,

    r.Region AS region_name, -- JOIN: Mengambil nama Region

    -- FEATURE ENGINEERING 2: Selisih Hari (Waiting Days) menggunakan fungsi waktu SQL
    CAST(julianday(substr(c.AppointmentDay, 1, 10)) - julianday(substr(c.ScheduledDay, 1, 10)) AS INTEGER) AS waiting_days,

    -- FEATURE ENGINEERING 3: Indikator Biner Datang
    CASE WHEN c.is_no_show = 'Yes' THEN 1 ELSE 0 END AS target_no_show,

    -- FEATURE ENGINEERING 4: Total Penyakit Bawaan
    (c.Hipertension + c.Diabetes + c.Alcoholism + c.Handcap) AS total_comorbidities

FROM CleanedData c
LEFT JOIN raw_regions r ON c.Neighbourhood = r.Lokasi; -- PENGGABUNGAN DATA (JOIN)
"""

# =======================================================
# TRANSFORMASI 3: Agregasi dan Perhitungan Metrik Analitik
# =======================================================
sql_aggregate = """
CREATE TABLE IF NOT EXISTS data_mart_region_metrics AS
SELECT
    region_name,
    COUNT(AppointmentID) AS total_kunjungan,
    SUM(target_no_show) AS total_pasien_bolos,
    ROUND((CAST(SUM(target_no_show) AS FLOAT) / COUNT(AppointmentID)) * 100, 2) AS no_show_rate_percent,
    ROUND(AVG(waiting_days), 1) AS rata_rata_hari_tunggu
FROM analytics_appointments
GROUP BY region_name
ORDER BY no_show_rate_percent DESC;
"""

try:
    # Eksekusi Transformasi
    cursor.execute("DROP TABLE IF EXISTS analytics_appointments;")
    cursor.execute("DROP TABLE IF EXISTS data_mart_region_metrics;")

    print("⏳ Menjalankan Query Transformasi (Cleaning, Join, Feature Engineering)...")
    cursor.execute(sql_transform)

    print("⏳ Menjalankan Query Agregasi Metrik Analitik...")
    cursor.execute(sql_aggregate)

    conn_lake.commit()
    print("✅ Transformasi ELT In-Warehouse Selesai!\n")

    # Menampilkan hasil Agregasi Metrik sebagai bukti
    print("📊 PRATINJAU HASIL AGREGASI (Tabel: data_mart_region_metrics):")
    df_metrics = pd.read_sql_query("SELECT * FROM data_mart_region_metrics", conn_lake)
    display(df_metrics)

except Exception as e:
    print(f"⚠️ Error saat Transformasi SQL: {e}")
finally:
    conn_lake.close()

--- [ELT - 7.2] TRANSFORMASI IN-WAREHOUSE (SQL) ---
⏳ Menjalankan Query Transformasi (Cleaning, Join, Feature Engineering)...
⏳ Menjalankan Query Agregasi Metrik Analitik...
✅ Transformasi ELT In-Warehouse Selesai!

📊 PRATINJAU HASIL AGREGASI (Tabel: data_mart_region_metrics):


,region_name,total_kunjungan,total_pasien_bolos,no_show_rate_percent,rata_rata_hari_tunggu
0,Região 1,9306,2018,21.68,10.0
1,Região 4,23913,5051,21.12,9.9
2,Região 3,16882,3532,20.92,10.2
3,Região 5,4060,826,20.34,8.1
4,Região 7,18018,3597,19.96,7.6
5,Região 6,9904,1955,19.74,12.9
6,Região 2,13405,2628,19.60,8.0
7,Região 8,7718,1465,18.98,18.8
8,Outros/Luar Vitória,1429,260,18.19,12.9
9,Região 9,5833,970,16.63,9.6
